In [66]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from datetime import datetime
import os

samples={"METRLA": 23974, "PEMSBAY": 36465, "PEMSD7M": 7589, "PEMS04": 10181, "PEMS07": 16921, "PEMS08": 10700}
target_batch_size = 16

def tpe(start, end, epochs, batch_size, dataset):
    dt1 = datetime.strptime(start, "%Y-%m-%d %H:%M:%S.%f")
    dt2 = datetime.strptime(end, "%Y-%m-%d %H:%M:%S.%f")
    time_diff = dt2 - dt1
    seconds = time_diff.total_seconds()
    
    time_per_ep = seconds / epochs
    batch_per_ep = samples[dataset] // batch_size + 1
    time_per_batch = time_per_ep / batch_per_ep
    
    time_per_ep_normalized = time_per_batch * (samples[dataset] // target_batch_size + 1)
    
    return time_per_ep_normalized

def tpe_auto(model_name, logs_dir="/data/dz/Torch-MTS/logs"):
    params_dict={}
    train_time_dict={}
    infer_time_dict={}
    for log_file in os.listdir(os.path.join(logs_dir, model_name)):
        dataset=log_file.split("-")[1]
        if dataset in samples:
            log_path=os.path.join(logs_dir, model_name, log_file)
            with open(log_path, "r") as log:
                lines=log.readlines()
                for line in lines:
                    if "batch_size" in line:
                        batch_size=int(line.split(":")[1].split(",")[0])
                    if "Total params" in line:
                        params=int(line.split(" ")[-1].strip().replace(",", "")) / 1000
                    if "Epoch 1 " in line:
                        start=line.split(" Epoch")[0]
                    if "Epoch" in line:
                        end=line.split(" Epoch")[0]
                        epochs=int(line.split(" ")[3])
                    if "Inference time" in line:
                        infer=float(line.split(" ")[2])
                        
            dt1 = datetime.strptime(start, "%Y-%m-%d %H:%M:%S.%f")
            dt2 = datetime.strptime(end, "%Y-%m-%d %H:%M:%S.%f")
            time_diff = dt2 - dt1
            seconds = time_diff.total_seconds()
            
            time_per_ep = seconds / epochs
            batch_per_ep = samples[dataset] // batch_size + 1
            time_per_batch = time_per_ep / batch_per_ep
            
            # print(dt1, dt2, epochs, batch_size, seconds)
            
            time_per_ep_normalized = time_per_batch * (samples[dataset] // target_batch_size + 1)
            train_time_dict[dataset]="%.2f" % time_per_ep_normalized
            infer_time_dict[dataset]="%.2f" % infer
            params_dict[dataset]="%.0f" % params
            
            # print(f"{dataset}: {time_per_ep_normalized:.2f}s")
    
    print(model_name, end=" & ")
    for key in samples.keys():
        print(f"{params_dict[key]}K & {train_time_dict[key]}s & {infer_time_dict[key]}s", end=" & ")
        if key=="PEMSD7M":
            print()
            print(model_name, end=" & ")
    print()
    for key in samples.keys():
        print(f"{key}\t\tParams: {params_dict[key]}K\t\tTrain: {train_time_dict[key]}s\t\tInfer: {infer_time_dict[key]}s")

In [67]:
tpe_auto("GRU")

GRU & 126K & 71.96s & 3.21s & 126K & 172.32s & 7.62s & 126K & 27.96s & 1.30s & 
GRU & 126K & 50.87s & 2.30s & 126K & 249.13s & 11.05s & 126K & 31.23s & 1.36s & 
METRLA		Params: 126K		Train: 71.96s		Infer: 3.21s
PEMSBAY		Params: 126K		Train: 172.32s		Infer: 7.62s
PEMSD7M		Params: 126K		Train: 27.96s		Infer: 1.30s
PEMS04		Params: 126K		Train: 50.87s		Infer: 2.30s
PEMS07		Params: 126K		Train: 249.13s		Infer: 11.05s
PEMS08		Params: 126K		Train: 31.23s		Infer: 1.36s


In [68]:
tpe_auto("STGCN")

STGCN & 246K & 81.39s & 2.63s & 306K & 210.90s & 6.35s & 257K & 33.86s & 0.97s & 
STGCN & 297K & 63.95s & 1.75s & 592K & 369.55s & 7.67s & 227K & 32.73s & 1.22s & 
METRLA		Params: 246K		Train: 81.39s		Infer: 2.63s
PEMSBAY		Params: 306K		Train: 210.90s		Infer: 6.35s
PEMSD7M		Params: 257K		Train: 33.86s		Infer: 0.97s
PEMS04		Params: 297K		Train: 63.95s		Infer: 1.75s
PEMS07		Params: 592K		Train: 369.55s		Infer: 7.67s
PEMS08		Params: 227K		Train: 32.73s		Infer: 1.22s


In [70]:
tpe_auto("DCRNN")

DCRNN & 0K & 529.94s & 15.48s & 0K & 1071.18s & 33.03s & 0K & 854.15s & 30.72s & 
DCRNN & 0K & 269.71s & 9.06s & 0K & 1330.26s & 48.46s & 0K & 211.56s & 6.61s & 
METRLA		Params: 0K		Train: 529.94s		Infer: 15.48s
PEMSBAY		Params: 0K		Train: 1071.18s		Infer: 33.03s
PEMSD7M		Params: 0K		Train: 854.15s		Infer: 30.72s
PEMS04		Params: 0K		Train: 269.71s		Infer: 9.06s
PEMS07		Params: 0K		Train: 1330.26s		Infer: 48.46s
PEMS08		Params: 0K		Train: 211.56s		Infer: 6.61s


In [71]:
tpe_auto("GWNET")

GWNET & 309K & 101.11s & 2.22s & 312K & 260.98s & 5.82s & 310K & 37.47s & 0.88s & 
GWNET & 311K & 72.94s & 1.79s & 323K & 445.99s & 11.26s & 309K & 37.90s & 0.92s & 
METRLA		Params: 309K		Train: 101.11s		Infer: 2.22s
PEMSBAY		Params: 312K		Train: 260.98s		Infer: 5.82s
PEMSD7M		Params: 310K		Train: 37.47s		Infer: 0.88s
PEMS04		Params: 311K		Train: 72.94s		Infer: 1.79s
PEMS07		Params: 323K		Train: 445.99s		Infer: 11.26s
PEMS08		Params: 309K		Train: 37.90s		Infer: 0.92s


In [72]:
tpe_auto("MTGNN")

MTGNN & 405K & 62.95s & 1.24s & 573K & 138.64s & 2.80s & 435K & 21.18s & 0.47s & 
MTGNN & 548K & 38.84s & 0.91s & 1368K & 212.79s & 5.13s & 353K & 28.48s & 0.51s & 
METRLA		Params: 405K		Train: 62.95s		Infer: 1.24s
PEMSBAY		Params: 573K		Train: 138.64s		Infer: 2.80s
PEMSD7M		Params: 435K		Train: 21.18s		Infer: 0.47s
PEMS04		Params: 548K		Train: 38.84s		Infer: 0.91s
PEMS07		Params: 1368K		Train: 212.79s		Infer: 5.13s
PEMS08		Params: 353K		Train: 28.48s		Infer: 0.51s


In [73]:
tpe_auto("AGCRN")

AGCRN & 752K & 87.15s & 2.68s & 753K & 197.93s & 5.38s & 752K & 31.67s & 1.02s & 
AGCRN & 749K & 58.13s & 1.69s & 755K & 342.73s & 9.89s & 150K & 38.44s & 1.18s & 
METRLA		Params: 752K		Train: 87.15s		Infer: 2.68s
PEMSBAY		Params: 753K		Train: 197.93s		Infer: 5.38s
PEMSD7M		Params: 752K		Train: 31.67s		Infer: 1.02s
PEMS04		Params: 749K		Train: 58.13s		Infer: 1.69s
PEMS07		Params: 755K		Train: 342.73s		Infer: 9.89s
PEMS08		Params: 150K		Train: 38.44s		Infer: 1.18s


In [74]:
tpe_auto("GTS")

GTS & 38377K & 190.48s & 4.86s & 58363K & 546.96s & 14.91s & 12158K & 54.12s & 1.70s & 
GTS & 16305K & 100.87s & 3.54s & 27088K & 1270.23s & 62.82s & 17136K & 69.07s & 1.79s & 
METRLA		Params: 38377K		Train: 190.48s		Infer: 4.86s
PEMSBAY		Params: 58363K		Train: 546.96s		Infer: 14.91s
PEMSD7M		Params: 12158K		Train: 54.12s		Infer: 1.70s
PEMS04		Params: 16305K		Train: 100.87s		Infer: 3.54s
PEMS07		Params: 27088K		Train: 1270.23s		Infer: 62.82s
PEMS08		Params: 17136K		Train: 69.07s		Infer: 1.79s


In [75]:
tpe_auto("STNorm")

STNorm & 224K & 64.80s & 0.88s & 284K & 167.36s & 2.31s & 235K & 23.37s & 0.34s & 
STNorm & 275K & 44.95s & 0.74s & 570K & 279.37s & 4.68s & 205K & 28.06s & 0.58s & 
METRLA		Params: 224K		Train: 64.80s		Infer: 0.88s
PEMSBAY		Params: 284K		Train: 167.36s		Infer: 2.31s
PEMSD7M		Params: 235K		Train: 23.37s		Infer: 0.34s
PEMS04		Params: 275K		Train: 44.95s		Infer: 0.74s
PEMS07		Params: 570K		Train: 279.37s		Infer: 4.68s
PEMS08		Params: 205K		Train: 28.06s		Infer: 0.58s


In [76]:
tpe_auto("STID")

STID & 118K & 12.06s & 0.75s & 122K & 19.25s & 0.62s & 118K & 4.71s & 0.21s & 
STID & 121K & 5.74s & 0.26s & 140K & 12.32s & 0.64s & 117K & 5.42s & 0.24s & 
METRLA		Params: 118K		Train: 12.06s		Infer: 0.75s
PEMSBAY		Params: 122K		Train: 19.25s		Infer: 0.62s
PEMSD7M		Params: 118K		Train: 4.71s		Infer: 0.21s
PEMS04		Params: 121K		Train: 5.74s		Infer: 0.26s
PEMS07		Params: 140K		Train: 12.32s		Infer: 0.64s
PEMS08		Params: 117K		Train: 5.42s		Infer: 0.24s


In [77]:
tpe_auto("STWA")

STWA & 375K & 113.90s & 7.41s & 447K & 284.30s & 19.25s & 388K & 41.21s & 2.78s & 
STWA & 436K & 76.69s & 5.70s & 786K & 678.93s & 60.99s & 353K & 47.88s & 3.80s & 
METRLA		Params: 375K		Train: 113.90s		Infer: 7.41s
PEMSBAY		Params: 447K		Train: 284.30s		Infer: 19.25s
PEMSD7M		Params: 388K		Train: 41.21s		Infer: 2.78s
PEMS04		Params: 436K		Train: 76.69s		Infer: 5.70s
PEMS07		Params: 786K		Train: 678.93s		Infer: 60.99s
PEMS08		Params: 353K		Train: 47.88s		Infer: 3.80s


In [78]:
tpe_auto("STDN")

STDN & 5971K & 459.43s & 10.39s & 6273K & 1479.01s & 25.91s & 6025K & 170.05s & 4.37s & 
STDN & 6227K & 312.12s & 7.98s & 4480K & 693.32s & 32.56s & 5876K & 200.60s & 5.71s & 
METRLA		Params: 5971K		Train: 459.43s		Infer: 10.39s
PEMSBAY		Params: 6273K		Train: 1479.01s		Infer: 25.91s
PEMSD7M		Params: 6025K		Train: 170.05s		Infer: 4.37s
PEMS04		Params: 6227K		Train: 312.12s		Infer: 7.98s
PEMS07		Params: 4480K		Train: 693.32s		Infer: 32.56s
PEMS08		Params: 5876K		Train: 200.60s		Infer: 5.71s


In [ ]:
# mtgnn

tpe("2024-04-22 00:04:14.890738", "2024-04-22 00:28:39.419358", epochs=93, batch_size=64, dataset="METRLA")
tpe("2024-04-22 00:05:04.922479", "2024-04-22 02:10:19.775262", epochs=141, batch_size=64, dataset="PEMS07") # 1,368,076
tpe("2024-04-22 08:49:15.157442", "2024-04-22 09:13:05.648680", epochs=200, batch_size=64, dataset="PEMS08") # 352,764

62.94848462738351

212.78507277971366

28.482102328035715

In [4]:
# agcrn

tpe("2024-04-21 14:24:24.949817", "2024-04-21 14:41:07.898297", epochs=46, batch_size=64, dataset="METRLA")
tpe("2024-04-21 18:25:22.967080", "2024-04-21 21:42:49.399928", epochs=138, batch_size=64, dataset="PEMS07")
tpe("2024-04-21 16:36:20.934715", "2024-04-21 16:53:24.047660", epochs=106, batch_size=64, dataset="PEMS08")

87.15476936347825

342.72698805534594

38.4356783583221

In [5]:
# stnorm

tpe("2024-04-22 16:23:32.422505", "2024-04-22 16:31:54.939038", epochs=31, batch_size=64, dataset="METRLA")
tpe("2024-04-22 16:51:38.337326", "2024-04-22 17:54:36.941621", epochs=54, batch_size=64, dataset="PEMS07")
tpe("2024-04-22 17:12:24.638797", "2024-04-22 17:17:41.767225", epochs=45, batch_size=64, dataset="PEMS08")

64.79761573909677

279.3685076247379

28.06334898571429

In [6]:
# stwa

tpe("2024-04-22 09:36:58.933838", "2024-04-22 10:43:25.468787", epochs=35, batch_size=16, dataset="METRLA")
tpe("2024-04-22 12:31:40.841417", "2024-04-23 01:09:49.244212", epochs=67, batch_size=16, dataset="PEMS07")
tpe("2024-04-22 12:20:59.590517", "2024-04-22 13:05:41.068785", epochs=56, batch_size=16, dataset="PEMS08")

113.90099854285714

678.931385

47.883540499999995

In [8]:
# stdn

tpe("2025-02-19 09:52:16.002744", "2025-02-19 10:53:33.929654", epochs=32, batch_size=64, dataset="METRLA")
tpe("2025-02-19 20:50:00.020876", "2025-02-20 01:44:39.715290", epochs=51, batch_size=32, dataset="PEMS07")
tpe("2025-02-19 20:46:29.580496", "2025-02-19 21:36:01.658822", epochs=59, batch_size=64, dataset="PEMS08")

459.43436984083337

693.3213495686275

200.5972962161017